In [3]:
import sys
from pathlib import Path

# Add project root to path for src imports
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data import (
    ARTIFACTS_DIR,
    CLEANED_DIR,
    calculate_pearson_correlations,
    load_and_preprocess_system,
    run_feature_selection_pipeline,
)

# Plot settings
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

In [4]:
# Step 1: Load system_4 from cleaned/
sys4_path = (
    CLEANED_DIR / "system_4.parquet"
    if (CLEANED_DIR / "system_4.parquet").exists()
    else CLEANED_DIR / "system_4.csv"
)
df_sys4 = load_and_preprocess_system(sys4_path)

print(f"Loaded system_4 daytime rows: {len(df_sys4):,}")
df_sys4.describe().T[["count", "mean", "std", "min", "50%", "max"]]

FileNotFoundError: [Errno 2] No such file or directory: '/home/raul/Desktop/solar-data-all/solarMLthesis/data/cleaned/system_4.csv'

In [ ]:
# Step 2: Pearson correlation with target (dc_power)
corr_series = calculate_pearson_correlations(df_sys4)
print("Pearson Correlation Coefficients (vs. dc_power):")
print(corr_series)

# Plot Heatmap
corr_matrix = df_sys4.select_dtypes(include=["number"]).corr()
plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1
)
plt.title("System 4 — Feature Correlation Matrix")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "system4_correlation_heatmap.png", dpi=300)
plt.show()

In [ ]:
numeric_cols = [
    c for c in df_sys4.select_dtypes(include=["number"]).columns if c != "dc_power"
]
n_cols = 2
n_rows = (len(numeric_cols) + 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    axes[idx].scatter(df_sys4[col], df_sys4["dc_power"], alpha=0.1, s=1)
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("dc_power")
    axes[idx].set_title(f"{col} vs dc_power (r = {corr_series.loc[col]:.2f})")

# Hide unused axes
for idx in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "system4_scatter_plots.png", dpi=300)
plt.show()

In [ ]:
# Step 3: Run selection and multi-system validation (10, 33, 50)
selected_feats, primary_corr, val_df = run_feature_selection_pipeline(
    cleaned_dir=CLEANED_DIR,
    artifacts_dir=ARTIFACTS_DIR,
    primary_system="system_4",
    validation_systems=["system_10", "system_33", "system_50"],
    threshold=0.3,
)

print(f"\nFinal Selected Features (|r| >= 0.3): {selected_feats}")
print("\nValidation across Systems (Pearson r):")
display(val_df)